# 멀티모달

- Vision · OCR · 이미지 생성

In [ ]:
%pip install -U pillow pytesseract easyocr

## Step 01 · 환경 설정

- Tesseract 엔진 설치: 
  - Windows [UB-Mannheim 설치 파일](https://github.com/UB-Mannheim/tesseract/wiki)
  - macOS `brew install tesseract tesseract-lang` 
  - Ubuntu `sudo apt install tesseract-ocr tesseract-ocr-kor`
- GPT Image 모델은 OpenAI 조직 인증(Organization Verification) 후 사용 가능

(선택) pip install -U google-genai

`.env`

```
OPENAI_API_KEY=sk-...  
GEMINI_API_KEY=...        # (선택) Nano Banana  
```

In [ ]:
import base64
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
VISION_MODEL = "gpt-6-luna"
IMAGE_MODEL = "gpt-image-2.5-flare"       # 빠른 생성 / 정밀 편집은 gpt-image-2.5-sunburst

OUT = Path("mm_out")
OUT.mkdir(exist_ok=True)
print("OpenAI:", "OK" if os.getenv("OPENAI_API_KEY") else "없음")

In [ ]:
# 실습용 샘플 영수증 이미지 만들기 (한글 폰트를 OS별로 탐색)
from PIL import Image, ImageDraw, ImageFont

FONT_CANDIDATES = [
    "C:/Windows/Fonts/Malgun Gothic.ttf",                               # Windows
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",                # macOS
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",           # Ubuntu (fonts-nanum)
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",    # Ubuntu (fonts-noto-cjk)
]
FONT_PATH = next((f for f in FONT_CANDIDATES if Path(f).exists()), None)
print("한글 폰트:", FONT_PATH or "없음 (기본 폰트 — 한글이 깨질 수 있음)")


def font(size):
    return ImageFont.truetype(FONT_PATH, size) if FONT_PATH else ImageFont.load_default()


def make_receipt(path):
    img = Image.new("RGB", (560, 720), "white")
    d = ImageDraw.Draw(img)
    lines = [
        ("소신 카페 강남점", 34), ("사업자 123-45-67890", 20), ("2026-09-29 12:41", 20), ("-" * 38, 20),
        ("아메리카노        2   9,000", 24), ("카페라떼          1   5,500", 24), ("치즈케이크        1   6,800", 24),
        ("-" * 38, 20), ("합계                 21,300", 28), ("카드 결제 (신한 ****1234)", 20),
    ]
    y = 40
    for text, size in lines:
        d.text((40, y), text, fill="black", font=font(size))
        y += size + 30
    img.save(path)
    return path


receipt_path = make_receipt(OUT / "receipt.png")
Image.open(receipt_path).size

## Step 02 · 이미지 입력 (Vision)

In [ ]:
# (1) URL로 전달
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg"

resp = client.responses.create(
    model=VISION_MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "이 이미지를 한 문장으로 설명해줘."},
            {"type": "input_image", "image_url": IMG_URL},
        ],
    }],
)
print(resp.output_text)

In [ ]:
# (2) 로컬 파일 → base64 data URL
def to_data_url(path):
    mime = "image/png" if str(path).endswith(".png") else "image/jpeg"
    return f"data:{mime};base64,{base64.b64encode(Path(path).read_bytes()).decode()}"


resp = client.responses.create(
    model=VISION_MODEL,
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": "이 영수증의 가게 이름과 합계 금액만 알려줘."},
        {"type": "input_image", "image_url": to_data_url(receipt_path), "detail": "original"},   # OCR은 original
    ]}],
)
print(resp.output_text)
print("입력 토큰:", resp.usage.input_tokens)

In [ ]:
# (3) detail에 따른 토큰 비교 — low는 저렴하지만 작은 글씨를 놓칠 수 있다
for detail in ["low", "high", "original"]:
    r = client.responses.create(
        model=VISION_MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": "합계 금액만 숫자로 답해."},
            {"type": "input_image", "image_url": to_data_url(receipt_path), "detail": detail},
        ]}],
    )
    print(f"{detail:>8}: 입력 토큰 {r.usage.input_tokens:>5} | 답변 {r.output_text!r}")

In [ ]:
# (4) 여러 이미지 비교
IMG_URL2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6e/Golde33443.jpg/640px-Golde33443.jpg"
resp = client.responses.create(
    model=VISION_MODEL,
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": "두 이미지의 공통점과 차이점을 각각 2가지씩 알려줘."},
        {"type": "input_image", "image_url": IMG_URL},
        {"type": "input_image", "image_url": IMG_URL2},
    ]}],
)
print(resp.output_text)


## Step 03 · 구조화 추출 

- 영수증 이미지 → Pydantic

In [ ]:
from pydantic import BaseModel, Field

class Item(BaseModel):
    name: str
    qty: int
    price: int = Field(description="해당 품목의 합계 금액(원)")

class Receipt(BaseModel):
    store: str
    date: str = Field(description="YYYY-MM-DD")
    items: list[Item]
    total: int

In [ ]:
parsed = client.responses.parse(
    model=VISION_MODEL,
    input=[
        {"role": "system", "content": "영수증 이미지에서 정보를 추출하세요. 보이지 않는 값은 추측하지 마세요."},
        {"role": "user", "content": [
            {"type": "input_image", "image_url": to_data_url(receipt_path), "detail": "original"},
        ]},
    ],
    text_format=Receipt,
)
receipt = parsed.output_parsed
print(receipt.model_dump_json(indent=2))

In [ ]:
# 코드로 검증 — 품목 합과 합계가 맞는지
print("품목 합계 일치:", sum(i.price for i in receipt.items) == receipt.total)

## Step 04 · LangChain 멀티모달 메시지

In [ ]:
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=VISION_MODEL)

msg = HumanMessage(content=[
    {"type": "text", "text": "이 영수증에서 가장 비싼 품목은?"},
    {"type": "image", "base64": base64.b64encode(receipt_path.read_bytes()).decode(), "mime_type": "image/png"},
])
print(llm.invoke([msg]).content)

In [ ]:
# URL 이미지도 같은 방식
msg2 = HumanMessage(content=[
    {"type": "text", "text": "이 동물의 품종을 추정하고 근거를 한 줄로 말해줘."},
    {"type": "image", "url": IMG_URL},
])
print(llm.invoke([msg2]).content)

In [ ]:
# 구조화 출력도 그대로 사용 가능
print(llm.with_structured_output(Receipt).invoke([HumanMessage(content=[
    {"type": "text", "text": "영수증 정보를 추출해."},
    {"type": "image", "base64": base64.b64encode(receipt_path.read_bytes()).decode(), "mime_type": "image/png"},
])]))

## Step 05 · 전통 OCR 

- Tesseract · EasyOCR

In [ ]:
import pytesseract
from PIL import ImageFilter, ImageOps

# Windows: pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
LANG = "kor+eng" if "kor" in pytesseract.get_languages() else "eng"
print("Tesseract 언어:", LANG)

raw_text = pytesseract.image_to_string(Image.open(receipt_path), lang=LANG)
print(raw_text)

In [ ]:
# 전처리 — 흑백 · 확대 · 이진화 · 선명화 후 재인식
def preprocess(path):
    img = ImageOps.grayscale(Image.open(path))
    img = img.resize((img.width * 2, img.height * 2))
    img = img.point(lambda p: 255 if p > 160 else 0)
    return img.filter(ImageFilter.SHARPEN)


pre = preprocess(receipt_path)
pre.save(OUT / "receipt_pre.png")
print(pytesseract.image_to_string(pre, lang=LANG, config="--psm 6"))   # psm 6: 한 덩어리 텍스트 블록


In [ ]:
# 단어별 위치 · 신뢰도 (Detection + Recognition 결과)
data = pytesseract.image_to_data(pre, lang=LANG, output_type=pytesseract.Output.DICT)
for text, conf, x, y in zip(data["text"], data["conf"], data["left"], data["top"]):
    if text.strip():
        print(f"({x:4d},{y:4d}) conf={float(conf):5.1f}  {text}")

In [ ]:
# EasyOCR — 딥러닝 기반 (CRAFT 검출 + CRNN 인식). 첫 실행 시 모델 다운로드
try:
    import easyocr
    reader = easyocr.Reader(["ko", "en"], gpu=False)
    for box, text, conf in reader.readtext(str(receipt_path)):
        print(f"{conf:.2f}  {text}")
except ImportError:
    print("easyocr 미설치 — pip install easyocr (PyTorch 포함 약 1GB)")

## Step 06 · OCR + LLM 

- 후처리 vs VLM 직접 추출

In [ ]:
# (1) OCR 텍스트 → LLM이 구조화 (이미지는 모델에 보내지 않음)
ocr_text = pytesseract.image_to_string(pre, lang=LANG, config="--psm 6")
from_ocr = client.responses.parse(
    model=VISION_MODEL,
    input=[
        {"role": "system", "content": "OCR로 추출한 영수증 텍스트입니다. 오인식된 글자는 문맥으로 바로잡아 추출하세요."},
        {"role": "user", "content": ocr_text},
    ],
    text_format=Receipt,
)

In [ ]:
# (2) 두 방식 비교
a, b = from_ocr.output_parsed, receipt
print(f"{'':12}{'OCR + LLM':>16}{'VLM 직접':>16}")
print(f"{'가게':12}{a.store:>16}{b.store:>16}")
print(f"{'품목 수':12}{len(a.items):>16}{len(b.items):>16}")
print(f"{'합계':12}{a.total:>16,}{b.total:>16,}")
print(f"{'입력 토큰':12}{from_ocr.usage.input_tokens:>16}{parsed.usage.input_tokens:>16}")

## Step 07 · 이미지 생성 

- Image API

In [ ]:
result = client.images.generate(
    model=IMAGE_MODEL,
    prompt=("미니멀한 플랫 일러스트. 따뜻한 베이지 배경의 카페 카운터 위에 아메리카노 한 잔과 치즈케이크. "
            "상단에 'SOSIN CAFE' 문구, 부드러운 자연광, 여백을 넉넉히."),
    size="1024x1024",
    quality="medium",            # low / medium / high
)
gen_path = OUT / "cafe_poster.png"
gen_path.write_bytes(base64.b64decode(result.data[0].b64_json))
print("저장:", gen_path, "| 사용량:", result.usage)

In [ ]:
# 이미지 편집 — 입력 이미지 + 지시문
with open(gen_path, "rb") as f:
    edited = client.images.edit(
        model="gpt-image-2.5-sunburst",          # 정밀 편집용
        image=[f],
        prompt="배경을 짙은 초록색으로 바꾸고, 케이크 옆에 작은 화분 하나를 추가해줘. 나머지는 그대로 유지.",
        input_fidelity="high",                    # 원본 특징 유지
    )
(OUT / "cafe_poster_edit.png").write_bytes(base64.b64decode(edited.data[0].b64_json))
print("편집본 저장 완료")

In [ ]:
# 투명 배경 · 여러 장 · 형식 지정
icons = client.images.generate(
    model=IMAGE_MODEL,
    prompt="커피콩 모양의 단순한 앱 아이콘, 두꺼운 외곽선, 단색",
    size="1024x1024",
    background="transparent",
    output_format="png",
    n=2,
)
for i, d in enumerate(icons.data):
    (OUT / f"icon_{i}.png").write_bytes(base64.b64decode(d.b64_json))
print(len(icons.data), "장 저장")

 ## Step 08 · 대화형 이미지 생성 
 
 - Responses API image_generation 도구

In [ ]:
def save_images(response, prefix):
    calls = [o for o in response.output if o.type == "image_generation_call"]
    for i, c in enumerate(calls):
        (OUT / f"{prefix}_{i}.png").write_bytes(base64.b64decode(c.result))
    return len(calls)


r1 = client.responses.create(
    model=VISION_MODEL,
    input="가을 한정 메뉴 '밤 라떼' 홍보용 인스타그램 이미지를 만들어줘. 따뜻한 톤, 제품 중심.",
    tools=[{"type": "image_generation", "model": IMAGE_MODEL, "size": "1024x1536"}],
)
print("생성:", save_images(r1, "latte_v1"), "장 |", r1.output_text)

In [ ]:
# 이전 응답을 이어서 수정 — 대화 맥락으로 반복 편집
r2 = client.responses.create(
    model=VISION_MODEL,
    previous_response_id=r1.id,
    input="좋아. 하단에 '9/30까지 20% 할인' 문구를 넣고 배경을 조금 더 밝게.",
    tools=[{"type": "image_generation", "model": IMAGE_MODEL, "size": "1024x1536"}],
)
print("수정:", save_images(r2, "latte_v2"), "장")

In [ ]:
# (선택) Google Nano Banana — pip install google-genai, GEMINI_API_KEY 필요
if os.getenv("GEMINI_API_KEY"):
    from google import genai

    g = genai.Client()
    inter = g.interactions.create(
        model="gemini-3.1-flash-image",          # Nano Banana 2
        input="가을 한정 '밤 라떼' 홍보 포스터, 따뜻한 톤, 한글 문구 '가을엔 밤 라떼'",
    )
    (OUT / "nano_banana.png").write_bytes(base64.b64decode(inter.output_image.data))
    print("Nano Banana 이미지 저장")
else:
    print("GEMINI_API_KEY 없음 — 건너뜀")

## GPT 5 Vision을 활용한 이미지 처리

### 순서
1. 이미지 분석 및 마케팅 아이디어, 짧은 스토리 생성
2. 차트/그래프 데이터 분석

In [ ]:
## 라이브러리 설치
%pip install openai pillow requests matplotlib dotenv

In [ ]:
from dotenv import load_dotenv
import openai
import base64
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import os
from datetime import datetime

load_dotenv()

# OpenAI 클라이언트 초기화
client = openai.OpenAI()

In [ ]:
plt.rc('font', family='Malgun Gothic')

In [ ]:
# helper functions

def encode_image_to_base64(image_path):
    """
    이미지 파일을 base64로 인코딩하는 함수
    """
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_image_with_gpt5(image_path, prompt, max_tokens=300):
    """
    GPT-5 API를 사용하여 이미지를 분석하는 함수
    """
    # 이미지를 base64로 인코딩
    base64_image = encode_image_to_base64(image_path)
    
    try:
        response = client.chat.completions.create(
            model="gpt-5", 
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/webp;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_completion_tokens=max_tokens
        )
        
        return response.choices[0].message.content
    
    except Exception as e:
        return f"오류가 발생했습니다: {str(e)}"

def display_image(image_path):
    """
    이미지를 화면에 표시하는 함수
    """
    img = Image.open(image_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"분석 대상 이미지: {os.path.basename(image_path)}")
    plt.show()



### 예제 1: 이미지 분석 및 마케팅 아이디어, 짧은 스토리 생성

- 이미지의 내용, 객체, 색상, 분위기 등을 분석
- 마케팅 카피 생성
- 이미지 기반 스토리텔링

In [ ]:
# 카페 내부 이미지 다운로드 후 경로 저장

image_path = '샘플 이미지 경로'

# url로 이미지 다운로드하는 코드?

In [ ]:
# 이미지 표시
display_image(image_path)

# 다양한 프롬프트로 이미지 분석
prompts = [
    "이 이미지를 자세히 분석하고 설명해주세요. 무엇이 보이고, 어떤 분위기인지 알려주세요.",
    
    "이 이미지를 보고 카페 마케팅 문구를 3개 만들어주세요. 감성적이고 매력적인 문구로 작성해주세요.",
    
    "이 장소에서 일어날 수 있는 짧은 스토리를 한국어로 작성해주세요. 등장인물과 상황을 포함해서요."
]

print("=" * 60)
print("🎯 GPT 5 이미지 분석 결과")
print("=" * 60)

for i, prompt in enumerate(prompts, 1):
    print(f"\n📝 분석 {i}: {prompt[:30]}...")
    print("-" * 50)
    
    # GPT 5로 이미지 분석
    result = analyze_image_with_gpt5(image_path, prompt, max_tokens=500)
    
    print(f"💡 결과:\n{result}")
    print("-" * 50)


### 예제 2: 차트/그래프 데이터 분석

- 차트의 데이터 포인트 읽기
- 트렌드 분석 및 해석
- 비즈니스 인사이트 도출
- 개선 방안 제안


In [ ]:
# 샘플 비즈니스 데이터 차트 생성
import numpy as np

months = ['1월', '2월', '3월', '4월', '5월', '6월', '7월', '8월', '9월', '10월', '11월', '12월']
revenue_2023 = [850, 920, 1150, 1080, 1200, 1350, 1500, 1420, 1300, 1450, 1600, 1800]
revenue_2024 = [950, 1020, 1250, 1180, 1400, 1550, 1650, 1520, 1480, 1650, 1750, 1950]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('2023-2024 비즈니스 성과 대시보드', fontsize=16, fontweight='bold')

# 1. 라인 차트 - 월별 매출 추이
ax1.plot(months, revenue_2023, marker='o', linewidth=2, label='2023년', color='#FF6B6B')
ax1.plot(months, revenue_2024, marker='s', linewidth=2, label='2024년', color='#4ECDC4')
ax1.set_title('월별 매출 추이 (만원)', fontsize=12, fontweight='bold')
ax1.set_ylabel('매출 (만원)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. 바 차트 - 분기별 매출 비교
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
q_2023 = [np.mean(revenue_2023[i:i+3]) for i in range(0, 12, 3)]
q_2024 = [np.mean(revenue_2024[i:i+3]) for i in range(0, 12, 3)]

x = np.arange(len(quarters))
width = 0.35
ax2.bar(x - width/2, q_2023, width, label='2023년', color='#FF6B6B', alpha=0.8)
ax2.bar(x + width/2, q_2024, width, label='2024년', color='#4ECDC4', alpha=0.8)
ax2.set_title('분기별 평균 매출 비교', fontsize=12, fontweight='bold')
ax2.set_ylabel('매출 (만원)')
ax2.set_xticks(x)
ax2.set_xticklabels(quarters)
ax2.legend()

# 3. 파이 차트 - 2024년 분기별 매출 비중
ax3.pie(q_2024, labels=quarters, autopct='%1.1f%%', startangle=90, 
        colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
ax3.set_title('2024년 분기별 매출 비중', fontsize=12, fontweight='bold')

# 4. 성장률 차트
growth_rate = [(r24/r23 - 1) * 100 for r23, r24 in zip(revenue_2023, revenue_2024)]
colors = ['#FF6B6B' if gr < 10 else '#4ECDC4' if gr < 20 else '#45B7D1' for gr in growth_rate]
ax4.bar(months, growth_rate, color=colors, alpha=0.8)
ax4.set_title('전년 대비 월별 성장률 (%)', fontsize=12, fontweight='bold')
ax4.set_ylabel('성장률 (%)')
ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax4.tick_params(axis='x', rotation=45)

ax4.text(0.02, 0.98, '색상별 성장률:\n🔴 < 10%\n🟢 10-20%\n🔵 > 20%', 
         transform=ax4.transAxes, verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()

# 차트 저장
chart_image_path = './business_dashboard.jpg'
plt.savefig(chart_image_path, dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 대시보드 분석

display_image(chart_image_path)

# 비즈니스 분석을 위한 프롬프트
business_prompts = [
    """
    이 비즈니스 대시보드를 자세히 분석해주세요. 
    각 차트에서 보이는 주요 수치와 트렌드를 읽어주시고, 
    전반적인 비즈니스 성과를 평가해주세요.
    """,
    
    """
    이 데이터를 바탕으로 다음을 분석해주세요:
    1. 가장 성과가 좋았던 시기와 그 이유
    2. 개선이 필요한 영역
    3. 향후 3개월 예측 및 권장사항
    """,
    
    """
    경영진을 위한 요약 보고서를 작성해주세요. 
    핵심 성과 지표, 주요 인사이트, 실행 가능한 액션 아이템을 포함해서 
    간결하고 임팩트 있게 작성해주세요.
    """
]

print("=" * 70)
print("📊 GPT 5 비즈니스 차트 분석 결과")
print("=" * 70)

for i, prompt in enumerate(business_prompts, 1):
    print(f"\n📈 분석 {i}: 비즈니스 인사이트 추출")
    print("-" * 60)
    
    # GPT 4V로 차트 분석
    result = analyze_image_with_gpt5(chart_image_path, prompt, max_tokens=800)
    
    print(f"💼 분석 결과:\n{result}")
    print("-" * 60)


### 이미지를 활용해 분석해볼만한 것

1. OCR (텍스트 추출)
   - 문서, 영수증, 명함에서 텍스트 추출
   - 손글씨 인식 및 디지털화

2. 제품 분석
   - 제품 이미지 분석 및 설명 생성
   - 경쟁사 제품 비교 분석

3. 의료 이미지 분석
   - X-ray, MRI 등 의료 이미지 보조 분석
   - 피부 질환 이미지 분석

4. 부동산 / 건축
   - 건물 외관 분석 및 평가
   - 인테리어 디자인 제안

5. 교육 콘텐츠
   - 수학 문제 해결
   - 과학 실험 결과 분석


당연하게도 프롬프트는 명확하고 구체적으로 작성해야 답변의 질이 높아짐.  
이미지 분석의 경우에는 이미지 품질도 큰 비중을 차지함.  

단, 결과를 맹신하지 말 것  
개인정보가 포함된 이미지는 업로드하지 말 것  
